# Load Pretrained Model

In [1]:
import os
import requests
url = "https://raw.githubusercontent.com/AdamClarkStandke/TransDiffTemp/refs/heads/main/gptStuff.py"
!touch "/content/gptStuff.py"
file_path = "/content/gptStuff.py"
response = requests.get(url, timeout=30)
response.raise_for_status()
text_data = response.text
with open(file_path, "w", encoding="utf-8") as file:
  file.write(text_data)

In [2]:
import torch
from gptStuff import GPTModel
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [3]:
!hf sync hf://buckets/DexHeim/testing ./model_path.pth


  A new version of huggingface_hub is available: 1.10.1 → 1.12.0

  Do you want to update now? [Y/n] (/usr/bin/python3 -m pip install -U huggingface_hub) n
  Skipped. You can update later with: /usr/bin/python3 -m pip install -U huggingface_hub

Scanning remote bucket (1 files)
Scanning local directory (0 files)
Comparing files (1 paths)
Sync plan: hf://buckets/DexHeim/testing -> ./model_path.pth
  Uploads: 0
  Downloads: 1
  Deletes: 0
  Skips: 0
Syncing...
Sync completed.


In [4]:
checkpoint = torch.load("model_path.pth/model_path.pth")
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

# Decoding strategies to control randomness